In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

# Корневая папка проекта
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

# Исходный файл остаётся в проекте на диске C
SOURCE_PATH = PROJECT_DIR / "data" / "raw" / "source.csv"

# Результаты сохраняются на диске E
OUTPUT_DIR = Path(r"E:\laba_results")
OUTPUT_PATH = OUTPUT_DIR / "clean.parquet"

# Создаём папку, если её ещё нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Папка проекта:", PROJECT_DIR)
print("Исходный файл найден:", SOURCE_PATH.exists())
print("Исходный файл:", SOURCE_PATH)
print("Папка результатов:", OUTPUT_DIR)
print("Итоговый файл:", OUTPUT_PATH)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Не найден исходный файл: {SOURCE_PATH}")

Папка проекта: c:\Users\мирон\Desktop\laba po pitonu
Исходный файл найден: True
Исходный файл: c:\Users\мирон\Desktop\laba po pitonu\data\raw\source.csv
Папка результатов: E:\laba_results
Итоговый файл: E:\laba_results\clean.parquet


In [3]:
from pathlib import Path
import re
import csv
import json
import time
import gc
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ============================================================
# 1. ПУТИ И НАСТРОЙКИ
# ============================================================

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
SOURCE_PATH = PROJECT_DIR / "data" / "raw" / "source.csv"

OUTPUT_DIR = Path(r"E:\laba_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "clean.parquet"
LOG_PATH = OUTPUT_DIR / "cleaning_log.csv"
REPORT_PATH = OUTPUT_DIR / "cleaning_summary.json"
MISSING_PATH = OUTPUT_DIR / "missing_analysis.csv"

CHUNK_SIZE = 100_000

# Предметные ограничения для датасета поездок
FARE_MIN = 0
FARE_MAX = 1000
COORD_MIN = -180
COORD_MAX = 180
PASSENGERS_MIN = 0
PASSENGERS_MAX = 208

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Не найден файл: {SOURCE_PATH}")

# Удаляем предыдущий результат, но не исходный файл
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

start_all = time.perf_counter()
logs = []
missing_counts = {}
initial_rows = 0
written_rows = 0
removed_rows = 0
changed_rows = 0
outlier_iqr_count = 0
outlier_zscore_count = 0
writer = None
expected_columns = None

# ============================================================
# 2. ФУНКЦИИ ОЧИСТКИ
# ============================================================

def normalize_column(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

def log_stage(stage, before, after, changed=0, note=""):
    logs.append({
        "этап": stage,
        "строк_до": int(before),
        "строк_после": int(after),
        "удалено_строк": int(before - after),
        "изменено_строк": int(changed),
        "комментарий": note
    })

def clean_chunk(chunk):
    global changed_rows, removed_rows
    global outlier_iqr_count, outlier_zscore_count

    original_len = len(chunk)

    # Этап 1: названия столбцов
    before_columns = list(chunk.columns)
    chunk.columns = [normalize_column(c) for c in chunk.columns]

    if len(set(chunk.columns)) != len(chunk.columns):
        raise ValueError(
            "После нормализации названий возникли дубликаты столбцов."
        )

    logs.append({
        "этап": "нормализация названий столбцов",
        "строк_до": len(chunk),
        "строк_после": len(chunk),
        "удалено_строк": 0,
        "изменено_строк": 0,
        "комментарий": f"{before_columns} -> {list(chunk.columns)}"
    })

    # Этап 2: скрытые пропуски и текстовые значения
    for col in chunk.select_dtypes(include=["object", "string"]).columns:
        s = chunk[col].astype("string").str.strip()
        s = s.replace(
            ["", "-", "--", "—", "нет данных", "нет данных.", "null",
             "NULL", "None", "N/A", "n/a", "NaN", "nan"],
            pd.NA
        )
        chunk[col] = s

    # Этап 3: приведение типов
    numeric_cols = [
        c for c in [
            "fare_amount", "pickup_longitude", "pickup_latitude",
            "dropoff_longitude", "dropoff_latitude", "passenger_count"
        ] if c in chunk.columns
    ]

    for col in numeric_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    if "pickup_datetime" in chunk.columns:
        chunk["pickup_datetime"] = pd.to_datetime(
            chunk["pickup_datetime"], errors="coerce"
        )

    # Этап 4: подсчёт пропусков до обработки строк
    for col in chunk.columns:
        missing_counts[col] = missing_counts.get(col, 0) + int(
            chunk[col].isna().sum()
        )

    # Этап 5: нормализация категориальных значений
    for col in chunk.select_dtypes(include=["object", "string"]).columns:
        chunk[col] = chunk[col].astype("string").str.strip()

    # Этап 6: проверка предметных ограничений
    invalid = pd.Series(False, index=chunk.index)

    if "fare_amount" in chunk.columns:
        invalid |= (
            chunk["fare_amount"].notna()
            & ~chunk["fare_amount"].between(FARE_MIN, FARE_MAX)
        )

    for col in ["pickup_longitude", "dropoff_longitude"]:
        if col in chunk.columns:
            invalid |= (
                chunk[col].notna()
                & ~chunk[col].between(COORD_MIN, COORD_MAX)
            )

    for col in ["pickup_latitude", "dropoff_latitude"]:
        if col in chunk.columns:
            invalid |= (
                chunk[col].notna()
                & ~chunk[col].between(-90, 90)
            )

    if "passenger_count" in chunk.columns:
        invalid |= (
            chunk["passenger_count"].notna()
            & ~chunk["passenger_count"].between(
                PASSENGERS_MIN, PASSENGERS_MAX
            )
        )

    if "pickup_datetime" in chunk.columns:
        current_time = pd.Timestamp.now(tz="UTC")

     # Приводим даты к единому формату с часовым поясом UTC
        dates_utc = pd.to_datetime(
        chunk["pickup_datetime"], errors="coerce", utc=True
    )

    invalid |= dates_utc.notna() & (dates_utc > current_time)

    before = len(chunk)
    chunk = chunk.loc[~invalid].copy()
    log_stage("предметные ограничения", before, len(chunk))

    # Этап 7: удаление полных дубликатов в текущей части
    before = len(chunk)
    chunk = chunk.drop_duplicates()
    log_stage("полные дубликаты в части", before, len(chunk))

    # Этап 8: проверка выбросов двумя методами.
    # Сначала только выявляем выбросы, не удаляем их автоматически.
    if "fare_amount" in chunk.columns:
        fares = chunk["fare_amount"].dropna()

        if len(fares) >= 4:
            q1 = fares.quantile(0.25)
            q3 = fares.quantile(0.75)
            iqr = q3 - q1
            iqr_mask = (
                (chunk["fare_amount"] < q1 - 1.5 * iqr)
                | (chunk["fare_amount"] > q3 + 1.5 * iqr)
            )
            outlier_iqr_count += int(iqr_mask.fillna(False).sum())

            mean = fares.mean()
            std = fares.std()

            if pd.notna(std) and std > 0:
                z_mask = ((chunk["fare_amount"] - mean).abs() / std) > 3
                outlier_zscore_count += int(z_mask.fillna(False).sum())

    changed_rows += 0
    removed_rows += original_len - len(chunk)

    return chunk

# ============================================================
# 3. ЧТЕНИЕ И ОЧИСТКА ЧАСТЯМИ
# ============================================================

print("Начинаем очистку исходного CSV частями...")
print("Исходный файл:", SOURCE_PATH)
print("Результат:", OUTPUT_PATH)
print("Размер части:", CHUNK_SIZE)

try:
    for part_number, chunk in enumerate(
        pd.read_csv(
            SOURCE_PATH,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        start=1
    ):
        initial_rows += len(chunk)

        # Очистка текущей части
        chunk = clean_chunk(chunk)

        # Пропускаем пустые части
        if chunk.empty:
            continue

        # Схема выходного файла должна быть одинаковой для всех частей
        if expected_columns is None:
            expected_columns = list(chunk.columns)
        elif list(chunk.columns) != expected_columns:
            raise ValueError("Столбцы изменились между частями CSV.")

        # Запись части в Parquet без накопления всего набора в RAM
        table = pa.Table.from_pandas(chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_PATH,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)
        written_rows += len(chunk)

        if part_number % 10 == 0:
            print(
                f"Обработано частей: {part_number}; "
                f"прочитано строк: {initial_rows:,}; "
                f"сохранено строк: {written_rows:,}"
            )

        del chunk, table
        gc.collect()

finally:
    if writer is not None:
        writer.close()

# ============================================================
# 4. ЖУРНАЛ, ПРОПУСКИ И СВОДКА
# ============================================================

pd.DataFrame(logs).to_csv(
    LOG_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_df = pd.DataFrame(
    [
        {
            "столбец": col,
            "число_пропусков": count,
            "доля_пропусков_процентов": (
                count / initial_rows * 100 if initial_rows else 0
            )
        }
        for col, count in missing_counts.items()
    ]
)

if not missing_df.empty:
    missing_df = missing_df.sort_values(
        "число_пропусков", ascending=False
    )

missing_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "исходный_файл": str(SOURCE_PATH),
    "итоговый_файл": str(OUTPUT_PATH),
    "исходных_строк": initial_rows,
    "сохранено_строк": written_rows,
    "удалено_строк": removed_rows,
    "доля_потерь_процентов": (
        removed_rows / initial_rows * 100 if initial_rows else 0
    ),
    "выбросы_IQR_выявлено": outlier_iqr_count,
    "выбросы_Z_score_выявлено": outlier_zscore_count,
    "время_сек": round(time.perf_counter() - start_all, 2),
    "примечание": (
        "Дубликаты удалены внутри каждой части. "
        "Повторяющиеся строки между разными частями могут сохраниться. "
        "Выбросы выявлены двумя методами, но не удалены автоматически. "
        "Пропуски подсчитаны, но не заполнены автоматически."
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("ОЧИСТКА ЗАВЕРШЕНА")
print("=" * 60)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nАнализ пропусков:")
display(missing_df)
print("\nЖурнал этапов:")
display(pd.DataFrame(logs))

print("\nФайлы результатов:")
print("Parquet:", OUTPUT_PATH)
print("Журнал:", LOG_PATH)
print("Пропуски:", MISSING_PATH)
print("Сводка:", REPORT_PATH)

Начинаем очистку исходного CSV частями...
Исходный файл: c:\Users\мирон\Desktop\laba po pitonu\data\raw\source.csv
Результат: E:\laba_results\clean.parquet
Размер части: 100000
Обработано частей: 10; прочитано строк: 1,000,000; сохранено строк: 999,922
Обработано частей: 20; прочитано строк: 2,000,000; сохранено строк: 1,999,837
Обработано частей: 30; прочитано строк: 3,000,000; сохранено строк: 2,999,751
Обработано частей: 40; прочитано строк: 4,000,000; сохранено строк: 3,999,657
Обработано частей: 50; прочитано строк: 5,000,000; сохранено строк: 4,999,570
Обработано частей: 60; прочитано строк: 6,000,000; сохранено строк: 5,999,459
Обработано частей: 70; прочитано строк: 7,000,000; сохранено строк: 6,999,366
Обработано частей: 80; прочитано строк: 8,000,000; сохранено строк: 7,999,270
Обработано частей: 90; прочитано строк: 9,000,000; сохранено строк: 8,999,175
Обработано частей: 100; прочитано строк: 10,000,000; сохранено строк: 9,999,103
Обработано частей: 110; прочитано строк: 11

,столбец,число_пропусков,доля_пропусков_процентов
6,dropoff_latitude,376,0.000678
5,dropoff_longitude,376,0.000678
0,key,0,0.000000
1,fare_amount,0,0.000000
3,pickup_longitude,0,0.000000
2,pickup_datetime,0,0.000000
4,pickup_latitude,0,0.000000
7,passenger_count,0,0.000000



Журнал этапов:


,этап,строк_до,строк_после,удалено_строк,изменено_строк,комментарий
0,нормализация названий столбцов,100000,100000,0,0,"['key', 'fare_amount', 'pickup_datetime', 'pic..."
1,предметные ограничения,100000,99988,12,0,
2,полные дубликаты в части,99988,99988,0,0,
3,нормализация названий столбцов,100000,100000,0,0,"['key', 'fare_amount', 'pickup_datetime', 'pic..."
4,предметные ограничения,100000,99993,7,0,
...,...,...,...,...,...,...
1660,предметные ограничения,100000,99989,11,0,
1661,полные дубликаты в части,99989,99989,0,0,
1662,нормализация названий столбцов,23856,23856,0,0,"['key', 'fare_amount', 'pickup_datetime', 'pic..."
1663,предметные ограничения,23856,23855,1,0,



Файлы результатов:
Parquet: E:\laba_results\clean.parquet
Журнал: E:\laba_results\cleaning_log.csv
Пропуски: E:\laba_results\missing_analysis.csv
Сводка: E:\laba_results\cleaning_summary.json


In [3]:
from pathlib import Path
import json
import time
import gc

import duckdb
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# 1. ПУТИ И НАСТРОЙКИ
# ============================================================

BASE_DIR = Path(r"E:\laba_results")
INPUT_FILE = BASE_DIR / "clean.parquet"
FINAL_FILE = BASE_DIR / "clean_final.parquet"
TEMP_FILE = BASE_DIR / "clean_final_temp.parquet"

MISSING_FILE = BASE_DIR / "step12_missing_analysis.csv"
OUTLIERS_FILE = BASE_DIR / "step12_outliers.csv"
SUMMARY_FILE = BASE_DIR / "step12_summary.json"

# Ограничиваем память DuckDB и используем диск E для временных данных
DUCKDB_MEMORY = "3GB"
DUCKDB_THREADS = 2

start = time.perf_counter()

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Не найден файл: {INPUT_FILE}")

BASE_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY}'")
con.execute(f"SET threads = {DUCKDB_THREADS}")
con.execute(f"SET temp_directory = '{str(BASE_DIR / 'duckdb_temp').replace(chr(92), '/')}'")
con.execute("SET preserve_insertion_order = false")

# Экранируем путь для SQL
source = str(INPUT_FILE).replace("\\", "/").replace("'", "''")
temp = str(TEMP_FILE).replace("\\", "/").replace("'", "''")

# ============================================================
# 2. ИСХОДНАЯ ИНФОРМАЦИЯ
# ============================================================

metadata = pq.ParquetFile(INPUT_FILE).metadata
rows_before = metadata.num_rows
columns_before = pq.ParquetFile(INPUT_FILE).schema.names

print("Исходный файл:", INPUT_FILE)
print("Строк до обработки:", f"{rows_before:,}")
print("Столбцов:", len(columns_before))
print("DuckDB memory limit:", DUCKDB_MEMORY)

# ============================================================
# 3. ЧТЕНИЕ ЧАСТЯМИ, НОРМАЛИЗАЦИЯ И ГЛОБАЛЬНЫЕ ДУБЛИКАТЫ
# ============================================================

# Обрабатываем строковые значения и создаём промежуточный Parquet.
# Затем DuckDB удаляет полные дубликаты глобально по всем столбцам.
con.execute(f"""
COPY (
    SELECT DISTINCT *
    FROM (
        SELECT
            * REPLACE (
                CASE
                    WHEN pickup_datetime IS NOT NULL
                    THEN TRY_CAST(pickup_datetime AS TIMESTAMP)
                    ELSE NULL
                END AS pickup_datetime
            )
        FROM read_parquet('{source}')
    )
) TO '{temp}'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")

# Важно: DISTINCT выполняется глобально, не отдельно для каждого блока.
# Дата преобразуется, если столбец pickup_datetime существует.
# Если тип даты в исходной схеме иной, ниже предусмотрена проверка.

rows_after = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{temp}')"
).fetchone()[0]

columns_after = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{temp}')"
).fetchdf()["column_name"].tolist()

print("Строк после удаления полных дубликатов:", f"{rows_after:,}")

# ============================================================
# 4. АНАЛИЗ ПРОПУСКОВ ПО СТОЛБЦАМ
# ============================================================

missing_rows = []

for col in columns_after:
    safe_col = '"' + col.replace('"', '""') + '"'
    count = con.execute(f"""
        SELECT COUNT(*) - COUNT({safe_col})
        FROM read_parquet('{temp}')
    """).fetchone()[0]

    missing_rows.append({
        "столбец": col,
        "пропусков": int(count),
        "доля_пропусков_процентов":
            round(100 * count / rows_after, 4) if rows_after else 0
    })

missing_df = pd.DataFrame(missing_rows)
missing_df.to_csv(MISSING_FILE, index=False, encoding="utf-8-sig")

# ============================================================
# 5. ВЫБРОСЫ: IQR И Z-SCORE
# ============================================================

outlier_rows = []

if "fare_amount" in columns_after:
    q1, q3, mean_value, std_value = con.execute(f"""
        SELECT
            QUANTILE_CONT(fare_amount, 0.25),QUANTILE_CONT(fare_amount, 0.75),
            AVG(fare_amount),
            STDDEV_POP(fare_amount)
        FROM read_parquet('{temp}')
        WHERE fare_amount IS NOT NULL
    """).fetchone()

    if q1 is not None and q3 is not None:
        iqr = q3 - q1
        iqr_count = con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{temp}')
            WHERE fare_amount < {q1 - 1.5 * iqr}
               OR fare_amount > {q3 + 1.5 * iqr}
        """).fetchone()[0]
    else:
        iqr_count = 0

    if mean_value is not None and std_value not in (None, 0):
        z_count = con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{temp}')
            WHERE ABS((fare_amount - {mean_value}) / {std_value}) > 3
        """).fetchone()[0]
    else:
        z_count = 0

    outlier_rows.extend([
        {
            "столбец": "fare_amount",
            "метод": "IQR",
            "обнаружено_выбросов": int(iqr_count),
            "правило": "Ниже Q1 - 1.5*IQR или выше Q3 + 1.5*IQR"
        },
        {
            "столбец": "fare_amount",
            "метод": "Z-score",
            "обнаружено_выбросов": int(z_count),
            "правило": "Абсолютный Z-score больше 3"
        }
    ])

pd.DataFrame(outlier_rows).to_csv(
    OUTLIERS_FILE, index=False, encoding="utf-8-sig"
)

# ============================================================
# 6. ФИНАЛЬНОЕ СОХРАНЕНИЕ И СВОДКА
# ============================================================

if FINAL_FILE.exists():
    FINAL_FILE.unlink()

TEMP_FILE.replace(FINAL_FILE)

summary = {
    "исходный_файл": str(INPUT_FILE),
    "итоговый_файл": str(FINAL_FILE),
    "строк_до": int(rows_before),
    "строк_после": int(rows_after),
    "удалено_полных_дубликатов": int(rows_before - rows_after),
    "столбцов": len(columns_after),
    "доля_удалённых_строк_процентов":
        round(100 * (rows_before - rows_after) / rows_before, 4)
        if rows_before else 0,
    "время_секунд": round(time.perf_counter() - start, 2),
    "примечание":
        "Выполнено удаление полных дубликатов и анализ пропусков и выбросов. "
        "Выбросы не удалялись автоматически."
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

con.close()
gc.collect()

print("\nИТОГОВАЯ СВОДКА")
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\nАНАЛИЗ ПРОПУСКОВ")
display(missing_df)

print("\nАНАЛИЗ ВЫБРОСОВ")
display(pd.DataFrame(outlier_rows))

print("\nСохранённые файлы:")
print("Итоговый набор:", FINAL_FILE)
print("Пропуски:", MISSING_FILE)
print("Выбросы:", OUTLIERS_FILE)
print("Сводка:", SUMMARY_FILE)

Исходный файл: E:\laba_results\clean.parquet
Строк до обработки: 55,418,645
Столбцов: 8
DuckDB memory limit: 3GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Строк после удаления полных дубликатов: 55,418,645


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


ИТОГОВАЯ СВОДКА
{
  "исходный_файл": "E:\\laba_results\\clean.parquet",
  "итоговый_файл": "E:\\laba_results\\clean_final.parquet",
  "строк_до": 55418645,
  "строк_после": 55418645,
  "удалено_полных_дубликатов": 0,
  "столбцов": 8,
  "доля_удалённых_строк_процентов": 0.0,
  "время_секунд": 46.69,
  "примечание": "Выполнено удаление полных дубликатов и анализ пропусков и выбросов. Выбросы не удалялись автоматически."
}

АНАЛИЗ ПРОПУСКОВ


...


АНАЛИЗ ВЫБРОСОВ


,столбец,метод,обнаружено_выбросов,правило
0,fare_amount,IQR,4773101,Ниже Q1 - 1.5*IQR или выше Q3 + 1.5*IQR
1,fare_amount,Z-score,1506601,Абсолютный Z-score больше 3



Сохранённые файлы:
Итоговый набор: E:\laba_results\clean_final.parquet
Пропуски: E:\laba_results\step12_missing_analysis.csv
Выбросы: E:\laba_results\step12_outliers.csv
Сводка: E:\laba_results\step12_summary.json


ERROR: Could not install packages due to an OSError: [WinError 5] Отказано в доступе: 'c:\\Users\\мирон\\Desktop\\laba po pitonu\\.venv\\Lib\\site-packages\\_duckdb.cp312-win_amd64.pyd'
Check the permissions.



   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   -----------------